# embedopt — Paper Experiments on Google Colab (A100)

This notebook reproduces the headline paper experiments end-to-end on a Colab A100. Each cell is independent; rerun any one of them after editing.

**Before you run:** *Runtime → Change runtime type → Hardware accelerator: GPU (A100 if available)*.

## 1. Verify GPU
If the cell prints `cuda` then sentence-transformers will automatically encode on the A100.

In [ ]:
import torch
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
    print('gpu mem (GB):', torch.cuda.get_device_properties(0).total_memory / 1e9)

## 2. Clone the repo
The shell driver installs the `[paper]` extra in the active Colab runtime, so the A100-enabled PyTorch environment remains visible.

In [ ]:
!test -d embeddings || git clone https://github.com/jorge-martinez-gil/embeddings.git
%cd embeddings
!python -m pip install -q -e '.[paper]'

## 3. Download the BEIR benchmark datasets (one-time, ~2 GB total)
The shell driver can download these automatically under `data/`. This cell is optional if you run the driver cells below.

In [ ]:
!pip install -q beir
from beir.util import download_and_unzip
for name in ('scifact', 'nfcorpus', 'arguana', 'fiqa', 'trec-covid'):
    download_and_unzip(
        f'https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{name}.zip',
        '/content',
    )

## 4. Smoke run (no GPU required)
Verifies the script end-to-end on the deterministic synthetic dataset using the `hashing` backbone. Should finish in seconds.

In [ ]:
!OUTPUT_DIR=results-smoke sh scripts/run_all.sh --smoke

## 5. Headline run — paper-grade backbones × BEIR
On a Colab A100 this typically completes in 30–90 minutes depending on dataset size. Drop datasets you don't need from the list to shorten.

In [ ]:
!BACKBONES="e5-base bge-base mxbai-large" \
 DATASETS="scifact nfcorpus arguana fiqa trec-covid" \
 OUTPUT_DIR=results \
 BATCH_SIZE=256 \
 PROFILE_REPEATS=20 \
 BOOTSTRAP=5000 \
 SIGNIFICANCE=5000 \
 sh scripts/run_all.sh

## 6. Plot the Pareto front (quality × bytes/vec) per (backbone, dataset)

In [ ]:
import json, glob
import pandas as pd
import matplotlib.pyplot as plt

frames = []
for csv_path in glob.glob('results/*candidates.csv'):
    frames.append(pd.read_csv(csv_path))
df = pd.concat(frames, ignore_index=True)

for (backbone, dataset), sub in df.groupby(['backbone', 'dataset']):
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(sub['bytes_per_vector'], sub['ndcg_at_10'], s=40)
    for _, row in sub.iterrows():
        ax.annotate(row['spec'], (row['bytes_per_vector'], row['ndcg_at_10']), fontsize=7)
    ax.set_xscale('log')
    ax.set_xlabel('bytes / vector')
    ax.set_ylabel('nDCG@10')
    ax.set_title(f'{backbone} on {dataset}')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 7. Hypervolume table
Higher hypervolume = better Pareto frontier. Use this single scalar to compare backbones across datasets.

In [ ]:
summary = json.loads(open('results/summary.json').read())
summary_df = pd.DataFrame(summary)
pivot = summary_df.pivot_table(index='backbone', columns='dataset', values='hypervolume')
pivot